# Sentiment Classification with RNN, LSTM, and GRU
### IMDB Movie Reviews - Binary Classification

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import re
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load and Preprocess IMDB Dataset

In [ ]:
from tensorflow.keras.datasets import imdb as keras_imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE   = 10000
MAX_LENGTH   = 256
EMBED_DIM    = 100
HIDDEN_DIM   = 128
NUM_LAYERS   = 1
BATCH_SIZE   = 64
LEARNING_RATE = 0.001
NUM_EPOCHS   = 10

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras_imdb.load_data(num_words=VOCAB_SIZE)

x_train_padded = pad_sequences(x_train_raw, maxlen=MAX_LENGTH, padding='post', truncating='post')
x_test_padded  = pad_sequences(x_test_raw,  maxlen=MAX_LENGTH, padding='post', truncating='post')

print(f"Training samples : {len(x_train_padded)}")
print(f"Test samples     : {len(x_test_padded)}")
print(f"Sample sequence  : {x_train_padded[0][:20]}")
print(f"Label            : {y_train_raw[0]}")

## 2. PyTorch Dataset and DataLoaders

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels    = torch.tensor(labels,    dtype=torch.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


train_dataset = IMDBDataset(x_train_padded, y_train_raw)
test_dataset  = IMDBDataset(x_test_padded,  y_test_raw)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches : {len(train_loader)}")
print(f"Test batches     : {len(test_loader)}")

## 3. Model Definitions
### 3.1 Vanilla RNN Classifier

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super(RNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn       = nn.RNN(embed_dim, hidden_dim, num_layers,
                                batch_first=True, nonlinearity='tanh')
        self.fc        = nn.Linear(hidden_dim, 1)
        self.sigmoid   = nn.Sigmoid()

    def forward(self, x):
        embedded        = self.embedding(x)
        _, hidden       = self.rnn(embedded)
        out             = self.fc(hidden[-1])
        return self.sigmoid(out).squeeze(1)

### 3.2 LSTM Classifier

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, num_layers,
                                 batch_first=True)
        self.fc        = nn.Linear(hidden_dim, 1)
        self.sigmoid   = nn.Sigmoid()

    def forward(self, x):
        embedded           = self.embedding(x)
        _, (hidden, _)     = self.lstm(embedded)
        out                = self.fc(hidden[-1])
        return self.sigmoid(out).squeeze(1)

### 3.3 GRU Classifier

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super(GRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru       = nn.GRU(embed_dim, hidden_dim, num_layers,
                                batch_first=True)
        self.fc        = nn.Linear(hidden_dim, 1)
        self.sigmoid   = nn.Sigmoid()

    def forward(self, x):
        embedded  = self.embedding(x)
        _, hidden = self.gru(embedded)
        out       = self.fc(hidden[-1])
        return self.sigmoid(out).squeeze(1)

## 4. Training and Evaluation Utilities

In [ ]:
def train_model(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for sequences, labels in loader:
        sequences, labels = sequences.to(device), labels.to(device)

        optimizer.zero_grad()
        predictions = model(sequences)
        loss        = criterion(predictions, labels)
        loss.backward()
        optimizer.step()

        total_loss    += loss.item() * len(labels)
        predicted_cls  = (predictions >= 0.5).long()
        total_correct += (predicted_cls == labels.long()).sum().item()
        total_samples += len(labels)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for sequences, labels in loader:
            sequences, labels = sequences.to(device), labels.to(device)

            predictions    = model(sequences)
            loss           = criterion(predictions, labels)
            predicted_cls  = (predictions >= 0.5).long()

            total_loss    += loss.item() * len(labels)
            total_correct += (predicted_cls == labels.long()).sum().item()
            total_samples += len(labels)

            all_preds.extend(predicted_cls.cpu().numpy())
            all_labels.extend(labels.cpu().long().numpy())

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy, all_preds, all_labels

In [ ]:
def run_training(model_class, model_name):
    print(f"\n{'='*50}")
    print(f"  Training {model_name}")
    print(f"{'='*50}")

    model     = model_class(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCELoss()

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc   = train_model(model, train_loader, optimizer, criterion)
        val_loss, val_acc, _, _ = evaluate_model(model, test_loader, criterion)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  "
              f"Train Loss: {tr_loss:.4f}  Train Acc: {tr_acc:.4f}  "
              f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

    return model, history

## 5. Train All Three Models

In [ ]:
rnn_model,  rnn_history  = run_training(RNNClassifier,  "Vanilla RNN")
lstm_model, lstm_history = run_training(LSTMClassifier, "LSTM")
gru_model,  gru_history  = run_training(GRUClassifier,  "GRU")

## 6. Final Evaluation on Test Set

In [ ]:
criterion = nn.BCELoss()

def compute_metrics(model, name):
    _, acc, preds, labels = evaluate_model(model, test_loader, criterion)
    precision = precision_score(labels, preds)
    recall    = recall_score(labels, preds)
    f1        = f1_score(labels, preds)
    cm        = confusion_matrix(labels, preds)
    print(f"\n--- {name} ---")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    return {"name": name, "accuracy": acc, "precision": precision,
            "recall": recall, "f1": f1, "cm": cm, "preds": preds, "labels": labels}

results = [
    compute_metrics(rnn_model,  "Vanilla RNN"),
    compute_metrics(lstm_model, "LSTM"),
    compute_metrics(gru_model,  "GRU"),
]

## 7. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Confusion Matrices", fontsize=15)

for ax, result in zip(axes, results):
    sns.heatmap(result["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=["Neg", "Pos"],
                yticklabels=["Neg", "Pos"], ax=ax)
    ax.set_title(result["name"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
histories = {
    "Vanilla RNN": rnn_history,
    "LSTM":        lstm_history,
    "GRU":         gru_history,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training Curves", fontsize=15)

for name, hist in histories.items():
    axes[0].plot(hist["train_loss"], label=name)
    axes[1].plot(hist["val_acc"],   label=name)

axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
metrics      = ["accuracy", "precision", "recall", "f1"]
model_names  = [r["name"] for r in results]
x            = np.arange(len(metrics))
width        = 0.25

fig, ax = plt.subplots(figsize=(10, 6))

for i, result in enumerate(results):
    values = [result[m] for m in metrics]
    ax.bar(x + i * width, values, width, label=result["name"])

ax.set_xticks(x + width)
ax.set_xticklabels(["Accuracy", "Precision", "Recall", "F1-Score"])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Model Comparison - All Metrics")
ax.legend()
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()